# 👁️ Lesson 5.4 — Computer Vision in Practice

**AI Crash Course for Absolute Beginners**

Computer vision is AI that understands images and video. In this notebook:
- Process images with OpenCV (resizing, filtering, edge detection)
- Run pretrained object detection with Hugging Face DETR
- Classify images with ResNet-50
- Segment images with a pretrained model
- Work with real-world images from URLs

> Run each cell with **Shift + Enter**. GPU not required.

In [ ]:
!pip install opencv-python-headless Pillow torch torchvision transformers matplotlib numpy --quiet

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import urllib.request
import torch

print(f"OpenCV version: {cv2.__version__}")
print(f"PyTorch version: {torch.__version__}")

---
## Part 1 — Image Fundamentals with OpenCV

In [ ]:
# Download a sample image
img_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/b/b9/Above_Gotham.jpg/640px-Above_Gotham.jpg"
urllib.request.urlretrieve(img_url, "city.jpg")

# OpenCV loads as BGR — convert to RGB for matplotlib
img_bgr = cv2.imread("city.jpg")
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

print(f"Image shape: {img_rgb.shape}  (H x W x C)")
print(f"dtype: {img_rgb.dtype}")
print(f"Value range: {img_rgb.min()} - {img_rgb.max()}")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(img_rgb)
axes[0].set_title("Original RGB")
axes[0].axis("off")

# Grayscale
gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
axes[1].imshow(gray, cmap="gray")
axes[1].set_title(f"Grayscale {gray.shape}")
axes[1].axis("off")

# Pixel value histogram
axes[2].hist(img_rgb.ravel(), bins=50, color="#1a6bc8", alpha=0.8)
axes[2].set_xlabel("Pixel value (0-255)")
axes[2].set_ylabel("Count")
axes[2].set_title("Pixel Intensity Distribution")

plt.tight_layout()
plt.show()

In [ ]:
# Image transformations
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

# 1. Resize
resized = cv2.resize(img_rgb, (320, 213))
axes[0,0].imshow(resized)
axes[0,0].set_title(f"Resized to 320x213")

# 2. Gaussian blur
blurred = cv2.GaussianBlur(img_rgb, (15, 15), 0)
axes[0,1].imshow(blurred)
axes[0,1].set_title("Gaussian Blur (15x15)")

# 3. Canny edge detection
edges = cv2.Canny(gray, 50, 150)
axes[0,2].imshow(edges, cmap="gray")
axes[0,2].set_title("Canny Edge Detection")

# 4. Brightness / contrast
bright = np.clip(img_rgb.astype(int) + 60, 0, 255).astype(np.uint8)
axes[1,0].imshow(bright)
axes[1,0].set_title("Brightness +60")

# 5. Horizontal flip
flipped = cv2.flip(img_rgb, 1)
axes[1,1].imshow(flipped)
axes[1,1].set_title("Horizontal Flip")

# 6. Channel separation
axes[1,2].imshow(img_rgb[:, :, 0], cmap="Reds")
axes[1,2].set_title("Red Channel Only")

for ax in axes.flatten(): ax.axis("off")
plt.suptitle("OpenCV Image Transformations", fontsize=13)
plt.tight_layout()
plt.show()

---
## Part 2 — Image Classification with ResNet-50

In [ ]:
from torchvision import models, transforms
import json

# Load ImageNet labels
url = "https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json"
urllib.request.urlretrieve(url, "imagenet_labels.json")
with open("imagenet_labels.json") as f:
    labels = json.load(f)

# Load pretrained model
resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
resnet.eval()

preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def classify_image(image_path_or_url):
    if image_path_or_url.startswith("http"):
        urllib.request.urlretrieve(image_path_or_url, "_temp.jpg")
        img = Image.open("_temp.jpg").convert("RGB")
    else:
        img = Image.open(image_path_or_url).convert("RGB")

    tensor = preprocess(img).unsqueeze(0)

    with torch.no_grad():
        logits = resnet(tensor)
        probs  = torch.softmax(logits, dim=1)[0]
        top5   = torch.topk(probs, 5)

    print("Top-5 predictions:")
    for score, idx in zip(top5.values, top5.indices):
        bar = "#" * int(score * 40)
        print(f"  {labels[idx]:<30} {score:.3f}  {bar}")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    ax1.imshow(img)
    ax1.axis("off")
    ax1.set_title("Input Image")

    names  = [labels[i] for i in top5.indices]
    scores = [s.item() for s in top5.values]
    ax2.barh(names[::-1], scores[::-1], color="#1a6bc8")
    ax2.set_xlabel("Probability")
    ax2.set_title("ResNet-50 Top-5 Predictions")
    plt.tight_layout()
    plt.show()


# Try it!
classify_image("https://upload.wikimedia.org/wikipedia/commons/thumb/4/43/Cute_dog.jpg/320px-Cute_dog.jpg")

---
## Part 3 — Object Detection with Hugging Face DETR

In [ ]:
from transformers import pipeline

# Facebook's DETR (Detection Transformer) — end-to-end object detection
detector = pipeline(
    "object-detection",
    model="facebook/detr-resnet-50",
    threshold=0.7
)

# Street scene image
street_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/a/af/Cara_de_quem_caiu_do_caminh%C3%A3o....jpg/640px-Cara_de_quem_caiu_do_caminh%C3%A3o....jpg"
urllib.request.urlretrieve(street_url, "street.jpg")

img = Image.open("street.jpg").convert("RGB")
results = detector(img)

print(f"Objects detected: {len(results)}")
for r in results:
    print(f"  {r['label']:<15} confidence={r['score']:.3f}  box={r['box']}")

In [ ]:
# Draw bounding boxes
img_arr = np.array(img)
img_draw = img_arr.copy()

COLORS = [(255,50,50), (50,50,255), (50,200,50), (255,165,0), (128,0,128)]

for i, r in enumerate(results):
    box   = r["box"]
    label = f"{r['label']} {r['score']:.2f}"
    color = COLORS[i % len(COLORS)]

    x1, y1, x2, y2 = box["xmin"], box["ymin"], box["xmax"], box["ymax"]
    cv2.rectangle(img_draw, (x1, y1), (x2, y2), color, 2)
    cv2.putText(img_draw, label, (x1, y1 - 5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

plt.figure(figsize=(10, 6))
plt.imshow(img_draw)
plt.axis("off")
plt.title(f"DETR Object Detection — {len(results)} objects found")
plt.tight_layout()
plt.show()

---
## Part 4 — Data Augmentation for Training

In [ ]:
from torchvision import transforms

augmentations = [
    ("Original",           transforms.Compose([transforms.ToTensor(), transforms.ToPILImage()])),
    ("Random Crop",        transforms.RandomCrop(300)),
    ("H-Flip",             transforms.RandomHorizontalFlip(p=1.0)),
    ("Color Jitter",       transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5)),
    ("Rotation 30°",       transforms.RandomRotation(30)),
    ("Grayscale",          transforms.Grayscale(num_output_channels=3)),
]

base_img = Image.open("city.jpg").convert("RGB").resize((400, 266))

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, (name, aug) in zip(axes.flatten(), augmentations):
    augmented = aug(base_img)
    ax.imshow(augmented)
    ax.set_title(name)
    ax.axis("off")

plt.suptitle("Data Augmentation — same image, different transformations", fontsize=12)
plt.tight_layout()
plt.show()
print("Augmentation multiplies your training data and improves generalisation.")

---
## Part 5 — Try Your Own Image

In [ ]:
# Upload your own image or paste any URL here
YOUR_IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/320px-Cat03.jpg"

print("=== Classification ===")
classify_image(YOUR_IMAGE_URL)

print("\n=== Object Detection ===")
urllib.request.urlretrieve(YOUR_IMAGE_URL, "_your_img.jpg")
your_img = Image.open("_your_img.jpg").convert("RGB")
detections = detector(your_img)
print(f"Found {len(detections)} objects:")
for d in detections:
    print(f"  {d['label']} ({d['score']:.2f})")

---
## Challenge Exercises

1. **Threshold tuning**: Change `threshold=0.7` in the detector to `0.3`. How many more (potentially wrong) detections appear?
2. **Webcam capture** (in Colab): Use `from google.colab.patches import cv2_imshow` and `cv2.VideoCapture(0)` to run detection on your webcam.
3. **Fine-tune DETR**: Try `facebook/detr-resnet-50` with a custom dataset using Hugging Face Trainer — same workflow as Lesson 4.5.
4. **Count objects**: Write a function that counts how many instances of each class are detected in an image.

---
## Summary: CV Task Toolbox

| Task | Free Model | Library |
|---|---|---|
| Classification | ResNet-50 | torchvision |
| Object Detection | DETR, YOLOv8 | transformers, ultralytics |
| Segmentation | SAM, SegFormer | transformers |
| OCR | TrOCR, Tesseract | transformers, pytesseract |
| Pose estimation | ViTPose | transformers |
| Image generation | Stable Diffusion | diffusers |

---
**You've completed the notebooks!** Return to the [course homepage](https://GirlEf.github.io/ai-crash-course/) to see what's next.